# Implementation and Performance Evaluation of Machine Learning Models for Financial Growth Analysis
**Student:** Paras Satpute | **Division/Roll:** A1-05 | **PRN:** 202401040027

## Problem statement
Estimate the observed annual revenue-growth percentage of large U.S. companies using firm scale and industry characteristics. This is a **small-sample tabular regression study**, not a stock-price prediction or causal financial model. The intended use is an exploratory analyst-support screen that may help compare firms for further investigation.

**Dataset:** 100-company U.S. revenue snapshot from the earlier web-scraped finance assignment.  
**Target:** `Revenue_Growth_Pct` (continuous, percentage points).  
**Predictors:** `Revenue_USD_Millions`, `Employees`, and `Industry`. Company name and headquarters are excluded as identifiers/location labels; target and target-derived fields are not used as predictors.

## Algorithms
Four required comparison families are used: Support Vector Regression (SVR), Multi-Layer Perceptron (MLP), Decision Tree Regression, and K-Nearest Neighbours Regression. Each is implemented both with Scikit-learn and from scratch. The scratch SVR is a linear epsilon-insensitive primal approximation; the scratch MLP is a one-hidden-layer neural network trained with backpropagation; scratch tree uses variance-reduction CART splits; scratch KNN uses Euclidean distance averaging.

> Important limitation: the source has only 100 cross-sectional companies. It does not contain multiple years per company, so this experiment estimates the recorded growth rate from contemporaneous firm descriptors. It cannot establish future growth forecasting or real-time trading performance.

In [ ]:
import base64, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
warnings.filterwarnings('ignore')
RANDOM_STATE=42
CSV_B64 = "UmFuayxDb21wYW55LEluZHVzdHJ5LFJldmVudWVfVVNEX01pbGxpb25zLFJldmVudWVfR3Jvd3RoX1BjdCxFbXBsb3llZXMsSGVhZHF1YXJ0ZXJzCjEsV2FsbWFydCxSZXRhaWwsNjgwOTg1LDUuMSwyMTAwMDAwLCJCZW50b252aWxsZSwgQXJrYW5zYXMiCjIsQW1hem9uLFJldGFpbCBhbmQgY2xvdWQgY29tcHV0aW5nLDYzNzk1OSwxMS4wLDE1NTYwMDAsIlNlYXR0bGUsIFdhc2hpbmd0b24iCjMsVW5pdGVkSGVhbHRoIEdyb3VwLEhlYWx0aGNhcmUsNDAwMjc4LDcuNyw0MDAwMDAsIk1pbm5ldG9ua2EsIE1pbm5lc290YSIKNCxBcHBsZSxUZWNobm9sb2d5LDM5MTAzNSwyLjAsMTY0MDAwLCJDdXBlcnRpbm8sIENhbGlmb3JuaWEiCjUsQ1ZTIEhlYWx0aCxIZWFsdGhjYXJlLDM3MjgwOSw0LjIsMjU5NTAwLCJXb29uc29ja2V0LCBSaG9kZSBJc2xhbmQiCjYsQmVya3NoaXJlIEhhdGhhd2F5LENvbmdsb21lcmF0ZSwzNzE0MzMsMS45LDM5MjQwMCwiT21haGEsIE5lYnJhc2thIgo3LEFscGhhYmV0LFRlY2hub2xvZ3kgYW5kIGNsb3VkIGNvbXB1dGluZywzNTAwMTgsMTMuOSwxODMzMjMsIk1vdW50YWluIFZpZXcsIENhbGlmb3JuaWEiCjgsRXh4b24gTW9iaWwsUGV0cm9sZXVtLDM0OTU5NSwxLjUsNjA5MDAsIlNwcmluZywgVGV4YXMiCjksTWNLZXNzb24gQ29ycG9yYXRpb24sSGVhbHRoY2FyZSwzMDg5NTEsMTEuNyw0ODAwMCwiSXJ2aW5nLCBUZXhhcyIKMTAsQ2VuY29yYSxQaGFybWFjeSB3aG9sZXNhbGUsMjkzOTU5LDEyLjEsNDQwMDAsIkNvbnNob2hvY2tlbiwgUGVubnN5bHZhbmlhIgoxMSxKUE1vcmdhbiBDaGFzZSxGaW5hbmNpYWxzLDI3ODkwNiwxNi41LDMxNzIzMywiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjEyLENvc3RjbyBXaG9sZXNhbGUsUmV0YWlsLDI1NDQ1Myw1LjAsMzMzMDAwLCJJc3NhcXVhaCwgV2FzaGluZ3RvbiIKMTMsQ2lnbmEsSGVhbHRoIGluc3VyYW5jZSwyNDcxMjEsMjYuNiw3MjM5OCwiQmxvb21maWVsZCwgQ29ubmVjdGljdXQiCjE0LE1pY3Jvc29mdCxUZWNobm9sb2d5IGFuZCBjbG91ZCBjb21wdXRpbmcsMjQ1MTIyLDE1LjcsMjI4MDAwLCJSZWRtb25kLCBXYXNoaW5ndG9uIgoxNSxDYXJkaW5hbCBIZWFsdGgsSGVhbHRoY2FyZSwyMjY4MjcsMTAuNiw0ODQxMSwiRHVibGluLCBPaGlvIgoxNixDaGV2cm9uLFBldHJvbGV1bSwyMDI3OTIsMC45LDQ1Mjk4LCJIb3VzdG9uLCBUZXhhcyIKMTcsQmFuayBvZiBBbWVyaWNhLEZpbmFuY2lhbHMsMTkyNDM0LDExLjksMjEzMTkzLCJDaGFybG90dGUsIE5vcnRoIENhcm9saW5hIgoxOCxHZW5lcmFsIE1vdG9ycyxBdXRvbW90aXZlLDE4NzQ0Miw5LjEsMTYyMDAwLCJEZXRyb2l0LCBNaWNoaWdhbiIKMTksRm9yZCBNb3RvcixBdXRvbW90aXZlLDE4NDk5Miw1LjAsMTcxMDAwLCJEZWFyYm9ybiwgTWljaGlnYW4iCjIwLEVsZXZhbmNlIEhlYWx0aCxIZWFsdGhjYXJlLDE3NzAxMSwzLjMsMTAzNjc5LCJJbmRpYW5hcG9saXMsIEluZGlhbmEiCjIxLENpdGlncm91cCxGaW5hbmNpYWxzLDE3MDc1Nyw4LjksMjI3ODU1LCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKMjIsTWV0YSBQbGF0Zm9ybXMsVGVjaG5vbG9neSwxNjQ1MDEsMjEuOSw3NDA2NywiTWVubG8gUGFyaywgQ2FsaWZvcm5pYSIKMjMsQ2VudGVuZSxIZWFsdGhjYXJlLDE2MzA3MSw1LjksNjA1MDAsIlN0LiBMb3VpcywgTWlzc291cmkiCjI0LFRoZSBIb21lIERlcG90LFJldGFpbCwxNTk1MTQsNC41LDQ3MDEwMCwiQXRsYW50YSwgR2VvcmdpYSIKMjUsRmFubmllIE1hZSxGaW5hbmNpYWxzLDE1MjY3MCw4LjEsODIwMCwiV2FzaGluZ3RvbiwgRC5DLiIKMjYsV2FsZ3JlZW5zIEJvb3RzIEFsbGlhbmNlLFBoYXJtYWNldXRpY2FsLDE0NzY1OCw2LjIsMjUyNTAwLCJEZWVyZmllbGQsIElsbGlub2lzIgoyNyxLcm9nZXIsUmV0YWlsLDE0NzEyMywtMS45LDQwOTAwMCwiQ2luY2lubmF0aSwgT2hpbyIKMjgsUGhpbGxpcHMgNjYsUGV0cm9sZXVtLDE0NTQ5NiwtMi45LDEzMjAwLCJIb3VzdG9uLCBUZXhhcyIKMjksTWFyYXRob24gUGV0cm9sZXVtLFBldHJvbGV1bSwxNDA0MTIsLTYuNiwxODMwMCwiRmluZGxheSwgT2hpbyIKMzAsVmVyaXpvbiBDb21tdW5pY2F0aW9ucyxUZWxlY29tbXVuaWNhdGlvbnMsMTM0Nzg4LDAuNiw5OTYwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjMxLE52aWRpYSxUZWNobm9sb2d5LDEzMDQ5NywxMTQuMiwzNjAwMCwiU2FudGEgQ2xhcmEsIENhbGlmb3JuaWEiCjMyLEdvbGRtYW4gU2FjaHMgR3JvdXAsRmluYW5jaWFscywxMjY4NTMsMTcuMCw0NjUwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjMzLFdlbGxzIEZhcmdvLEZpbmFuY2lhbHMsMTI1Mzk3LDguNywyMTc1MDIsIlNhbiBGcmFuY2lzY28sIENhbGlmb3JuaWEiCjM0LFZhbGVybyBFbmVyZ3ksUGV0cm9sZXVtLDEyMzk3NCwtMTAuOCw5OTEwLCJTYW4gQW50b25pbywgVGV4YXMiCjM1LENvbWNhc3QsVGVsZWNvbW11bmljYXRpb25zLDEyMzczMSwxLjgsMTgyMDAwLCJQaGlsYWRlbHBoaWEsIFBlbm5zeWx2YW5pYSIKMzYsU3RhdGUgRmFybSBJbnN1cmFuY2UsRmluYW5jaWFscywxMjI5NTEsMTguMCw2NzM4MSwiQmxvb21pbmd0b24sIElsbGlub2lzIgozNyxBVCZULENvbmdsb21lcmF0ZSBhbmQgdGVsZWNvbW11bmljYXRpb25zLDEyMjMzNiwtMC4xLDE0MDk5MCwiRGFsbGFzLCBUZXhhcyIKMzgsRnJlZGRpZSBNYWMsRmluYW5jaWFscywxMjIwNTIsMTMuMCw4MDkwLCJNY0xlYW4sIFZpcmdpbmlhIgozOSxIdW1hbmEsSGVhbHRoIGluc3VyYW5jZSwxMTc3NjEsMTAuNyw2NTY4MCwiTG91aXN2aWxsZSwgS2VudHVja3kiCjQwLE1vcmdhbiBTdGFubGV5LEZpbmFuY2lhbHMsMTA3Mjg1LDExLjUsODA0NzgsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo0MSxUYXJnZXQsUmV0YWlsLDEwNjU2NiwtMC44LDQ0MDAwMCwiTWlubmVhcG9saXMsIE1pbm5lc290YSIKNDIsU3RvbmVYIEdyb3VwLEZpbmFuY2lhbHMsOTk4ODgsNjQuMSw0NTAwLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKNDMsVGVzbGEsQXV0b21vdGl2ZSBhbmQgZW5lcmd5LDk3NjkwLDAuOSwxMjU2NjUsIkF1c3RpbiwgVGV4YXMiCjQ0LERlbGwgVGVjaG5vbG9naWVzLFRlY2hub2xvZ3ksOTU1NjcsOC4xLDEwODAwMCwiUm91bmQgUm9jaywgVGV4YXMiCjQ1LFBlcHNpQ28sQmV2ZXJhZ2UsOTE4NTQsMC40LDMxOTAwMCwiUHVyY2hhc2UsIE5ldyBZb3JrIgo0NixXYWx0IERpc25leSxNZWRpYSw5MTM2MSwyLjgsMjA1MDQwLCJCdXJiYW5rLCBDYWxpZm9ybmlhIgo0NyxVbml0ZWQgUGFyY2VsIFNlcnZpY2UsVHJhbnNwb3J0YXRpb24sOTEwNzAsMC4xLDM3MjE4MCwiQXRsYW50YSwgR2VvcmdpYSIKNDgsSm9obnNvbiAmIEpvaG5zb24sUGhhcm1hY2V1dGljYWwsODg4MjEsLTYuNywxMzgxMDAsIk5ldyBCcnVuc3dpY2ssIE5ldyBKZXJzZXkiCjQ5LEZlZEV4LFRyYW5zcG9ydGF0aW9uLDg3NjkzLC0yLjcsNDIyMTAwLCJNZW1waGlzLCBUZW5uZXNzZWUiCjUwLEFyY2hlciBEYW5pZWxzIE1pZGxhbmQsRm9vZCBpbmR1c3RyeSw4NTUzMCwtOC45LDQzMjEzLCJDaGljYWdvLCBJbGxpbm9pcyIKNTEsUHJvY3RlciAmIEdhbWJsZSxDb25zdW1lciBwcm9kdWN0cyBtYW51ZmFjdHVyaW5nLDg0MDM5LDIuNSwxMDgwMDAsIkNpbmNpbm5hdGksIE9oaW8iCjUyLExvd2UncyxSZXRhaWwsODM2NzQsLTMuMSwyMTU1MDAsIk1vb3Jlc3ZpbGxlLCBOb3J0aCBDYXJvbGluYSIKNTMsRW5lcmd5IFRyYW5zZmVyIFBhcnRuZXJzLFBldHJvbGV1bSw4MjY3MSw1LjIsMTYyNDgsIkRhbGxhcywgVGV4YXMiCjU0LFJUWCxBZXJvc3BhY2UgYW5kIGRlZmVuc2UsODA3MzksMTcuMSwxODYwMDAsIkFybGluZ3RvbiBDb3VudHksIFZpcmdpbmlhIgo1NSxBbGJlcnRzb25zLFJldGFpbCw3OTIzOCwyLjAsMTk2NjUwLCJCb2lzZSwgSWRhaG8iCjU2LFN5c2NvLEZvb2QgcHJvY2Vzc2luZyw3ODg0NCwzLjMsNzYwMDAsIkhvdXN0b24sIFRleGFzIgo1NyxQcm9ncmVzc2l2ZSxJbnN1cmFuY2UsNzUzNzIsMjEuNCw2NjMwOCwiTWF5ZmllbGQgVmlsbGFnZSwgT2hpbyIKNTgsQW1lcmljYW4gRXhwcmVzcyxGaW5hbmNpYWxzLDc0MjAxLDEwLjEsNzUxMDAsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo1OSxMb2NraGVlZCBNYXJ0aW4sQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDcxMDQzLDUuMSwxMjEwMDAsIkJldGhlc2RhLCBNYXJ5bGFuZCIKNjAsTWV0TGlmZSxGaW5hbmNpYWxzLDcwOTg2LDYuMSw0NTAwMCwiTmV3IFlvcmsgQ2l0eSwgTmV3IFlvcmsiCjYxLEhDQSBIZWFsdGhjYXJlLEhlYWx0aGNhcmUsNzA2MDMsOC43LDI3MTAwMCwiTmFzaHZpbGxlLCBUZW5uZXNzZWUiCjYyLFBydWRlbnRpYWwgRmluYW5jaWFsLEZpbmFuY2lhbHMsNzA0MDUsMzAuNCwzNzkzNiwiTmV3YXJrLCBOZXcgSmVyc2V5Igo2MyxCb2VpbmcsQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDY2NTE3LC0xNC41LDE3MjAwMCwiQXJsaW5ndG9uIENvdW50eSwgVmlyZ2luaWEiCjY0LENhdGVycGlsbGFyLE1hY2hpbmVyeSw2NDgwOSwtMy40LDExMjkwMCwiSXJ2aW5nLCBUZXhhcyIKNjUsTWVyY2ssUGhhcm1hY2V1dGljYWwgaW5kdXN0cnksNjQxNjgsNi43LDc0MDAwLCJLZW5pbHdvcnRoLCBOZXcgSmVyc2V5Igo2NixBbGxzdGF0ZSxJbnN1cmFuY2UsNjQxMDYsMTIuMyw1NTIwMCwiTm9ydGhmaWVsZCBUb3duc2hpcCwgSWxsaW5vaXMiCjY3LFBmaXplcixQaGFybWFjZXV0aWNhbCBpbmR1c3RyeSw2MzYyNyw4LjgsODEwMDAsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo2OCxJQk0sVGVjaG5vbG9neSBhbmQgY2xvdWQgY29tcHV0aW5nLDYyNzUzLDEuNCwyODQ1MDAsIkFybW9uaywgTmV3IFlvcmsiCjY5LE5ldyBZb3JrIExpZmUgSW5zdXJhbmNlIENvbXBhbnksSW5zdXJhbmNlLDYyNjM5LDE1LjMsMTUxMzEsIk5ldyBZb3JrIENpdHksIE5ldyBZb3JrIgo3MCxEZWx0YSBBaXIgTGluZXMsQWlybGluZSw2MTY0Myw2LjIsMTAzMDAwLCJBdGxhbnRhLCBHZW9yZ2lhIgo3MSxQdWJsaXggU3VwZXIgTWFya2V0cyxSZXRhaWwsNjAxNzcsNC42LDI1NTAwMCwiTGFrZWxhbmQsIEZsb3JpZGEiCjcyLE5hdGlvbndpZGUgTXV0dWFsIEluc3VyYW5jZSBDb21wYW55LEZpbmFuY2lhbHMsNTg2NDYsNy40LDIyNDUzLCJDb2x1bWJ1cywgT2hpbyIKNzMsVEQgU3lubmV4LEluZm9ybWF0aW9uIFRlY2hub2xvZ3ksNTg0NTIsMS42LDI1NzUwLCJDbGVhcndhdGVyLCBGbG9yaWRhIgo3NCxVbml0ZWQgQWlybGluZXMgSG9sZGluZ3MsQWlybGluZSw1NzA2Myw2LjIsMTA3MzAwLCJDaGljYWdvLCBJbGxpbm9pcyIKNzUsQ29ub2NvUGhpbGxpcHMsUGV0cm9sZXVtLDU2OTUzLC0yLjgsMTE4MDAsIkhvdXN0b24sIFRleGFzIgo3NixUSlgsUmV0YWlsLDU2MzYwLDQuMCwzNjQwMDAsIkZyYW1pbmdoYW0sIE1hc3NhY2h1c2V0dHMiCjc3LEFiYlZpZSxQaGFybWFjZXV0aWNhbCw1NjMzNCwzLjcsNTUwMDAsIkxha2UgQmx1ZmYsIElsbGlub2lzIgo3OCxFbnRlcnByaXNlIFByb2R1Y3RzIFBhcnRuZXJzLFBldHJvbGV1bSw1NjIxOSwxMy4xLDc4MDAsIkhvdXN0b24sIFRleGFzIgo3OSxDaGFydGVyIENvbW11bmljYXRpb25zLFRlbGVjb21tdW5pY2F0aW9ucyw1NTA4NSwwLjksOTQ1MDAsIlN0YW1mb3JkLCBDb25uZWN0aWN1dCIKODAsUGVyZm9ybWFuY2UgRm9vZCBHcm91cCxGb29kIHByb2Nlc3NpbmcsNTQ2ODEsMi41LDM2ODE1LCJSaWNobW9uZCwgVmlyZ2luaWEiCjgxLEFtZXJpY2FuIEFpcmxpbmVzIEdyb3VwLEFpcmxpbmUsNTQyMTEsMi43LDEzMzMwMCwiRm9ydCBXb3J0aCwgVGV4YXMiCjgyLENhcGl0YWwgT25lIEZpbmFuY2lhbCxGaW5hbmNpYWxzLDUzOTM4LDkuMCw1MjYwMCwiVHlzb25zLCBWaXJnaW5pYSIKODMsQ2lzY28gU3lzdGVtcyxUZWxlY29tIGhhcmR3YXJlIG1hbnVmYWN0dXJpbmcsNTM4MDMsLTUuNiw5MDQwMCwiU2FuIEpvc2UsIENhbGlmb3JuaWEiCjg0LEhQLFRlY2hub2xvZ3ksNTM1NTksLTAuMyw1ODAwMCwiUGFsbyBBbHRvLCBDYWxpZm9ybmlhIgo4NSxUeXNvbiBGb29kcyxGb29kIHByb2Nlc3NpbmcsNTMzMDksMC44LDEzODAwMCwiU3ByaW5nZGFsZSwgQXJrYW5zYXMiCjg2LEludGVsLFRlY2hub2xvZ3ksNTMxMDEsLTIuMSwxMDg5MDAsIlNhbnRhIENsYXJhLCBDYWxpZm9ybmlhIgo4NyxPcmFjbGUsVGVjaG5vbG9neSw1Mjk2MSw2LjAsMTU5MDAwLCJBdXN0aW4sIFRleGFzIgo4OCxCcm9hZGNvbSxUZWNobm9sb2d5LDUyNDMyLDQ2LjQsMzcwMDAsIlBhbG8gQWx0bywgQ2FsaWZvcm5pYSIKODksRGVlcmUsQWdyaWN1bHR1cmUgbWFudWZhY3R1cmluZyw1MTcxNiwtMTUuNiw1NTUyNCwiTW9saW5lLCBJbGxpbm9pcyIKOTAsTmlrZSxBcHBhcmVsLDUxMzYxLDAuMyw3ODQwMCwiQmVhdmVydG9uLCBPcmVnb24iCjkxLExpYmVydHkgTXV0dWFsIEluc3VyYW5jZSBHcm91cCxJbnN1cmFuY2UsNTA0MjksLTQuMSw0MDAwMCwiQm9zdG9uLCBNYXNzYWNodXNldHRzIgo5MixQbGFpbnMgR1AgSG9sZGluZ3MsUGV0cm9sZXVtLDUwMDczLDIuOCw0MjAwLCJIb3VzdG9uLCBUZXhhcyIKOTMsVVNBQSxJbnN1cmFuY2UsNDg1NjAsMTQuMywzODAxOCwiU2FuIEFudG9uaW8sIFRleGFzIgo5NCxCcmlzdG9sLU15ZXJzIFNxdWliYixQaGFybWFjZXV0aWNhbCw0ODMwMCw3LjMsMzQxMDAsIk5ldyBZb3JrLCBOZXcgWW9yayIKOTUsSW5ncmFtIE1pY3JvIEhvbGRpbmcsVGVjaG5vbG9neSw0Nzk4NCwtMC4xLDI2MTI1LCJJcnZpbmUsIENhbGlmb3JuaWEiCjk2LEdlbmVyYWwgRHluYW1pY3MsQWVyb3NwYWNlIGFuZCBkZWZlbnNlLDQ3NzE2LDEyLjksMTE3MDAwLCJSZXN0b24sIFZpcmdpbmlhIgo5NyxDb2NhLUNvbGEsQmV2ZXJhZ2UsNDcwNjEsMi45LDY5NzAwLCJBdGxhbnRhLCBHZW9yZ2lhIgo5OCxUSUFBLEZpbmFuY2lhbHMsNDY5NDYsMi42LDE1NjIzLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKOTksVHJhdmVsZXJzLEluc3VyYW5jZSw0NjQyMywxMi4yLDM0MDAwLCJOZXcgWW9yayBDaXR5LCBOZXcgWW9yayIKMTAwLEVsaSBMaWxseSxQaGFybWFjZXV0aWNhbCw0NTA0MywzMi4wLDQ3MDAwLCJJbmRpYW5hcG9saXMsIEluZGlhbmEiCg=="
df=pd.read_csv(io.BytesIO(base64.b64decode(CSV_B64)))
print(df.shape)
display(df.head())

## 1. Statistical summary and exploration

In [ ]:
print(df.info())
display(df[['Revenue_USD_Millions','Revenue_Growth_Pct','Employees']].describe().T)
print('Missing values:',df.isnull().sum().sum())
print('Industries:',df['Industry'].nunique())
display(df.groupby('Industry')['Revenue_Growth_Pct'].agg(['count','mean','median','std']).sort_values('count',ascending=False).head(15))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].hist(df['Revenue_Growth_Pct'],bins=18,edgecolor='black')
ax[0].set_title('Revenue growth distribution'); ax[0].set_xlabel('Growth (%)')
ax[1].scatter(df['Revenue_USD_Millions'],df['Revenue_Growth_Pct'])
ax[1].set_title('Revenue vs growth'); ax[1].set_xlabel('Revenue (USD millions)'); ax[1].set_ylabel('Growth (%)')
plt.tight_layout(); plt.show()
print(df[['Revenue_USD_Millions','Revenue_Growth_Pct','Employees']].corr())

## 2. Target, predictors, split and preprocessing

An 80:20 random split (seed 42) is used. All transformations are fitted on training data only. Numeric predictors are standardized and industry is one-hot encoded. The same transformed matrix is supplied to all scratch models. The library models use pipelines so scaling/encoding is repeated inside each CV fold without leakage.

In [ ]:
target='Revenue_Growth_Pct'
X=df[['Revenue_USD_Millions','Employees','Industry']].copy()
y=df[target].astype(float)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
num_cols=['Revenue_USD_Millions','Employees']; cat_cols=['Industry']
pre=ColumnTransformer([
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat_cols)
])
X_train_s=pre.fit_transform(X_train)
X_test_s=pre.transform(X_test)
print('Train:',X_train.shape,'Test:',X_test.shape)
print('Train target summary:'); display(y_train.describe())
print('Test target summary:'); display(y_test.describe())

## 3. From-scratch algorithms

- **SVR:** linear epsilon-insensitive regression. It minimizes an L2 weight penalty plus epsilon-insensitive absolute error using batch subgradient descent. This is not a full kernel-SVR solver.
- **MLP:** one hidden tanh layer, linear output and backpropagation with L2 regularization.
- **Decision Tree:** recursive CART regression using reduction in within-node squared error (variance/MSE impurity).
- **KNN:** Euclidean distance and uniform or inverse-distance weighted average of the nearest targets.

In [ ]:
def variance_impurity(y):
    return float(np.mean((y-np.mean(y))**2)) if len(y) else 0.0

class ScratchTreeRegressor:
    def __init__(self,max_depth=2,min_samples_leaf=5,max_thresholds=20):
        self.max_depth=max_depth; self.min_samples_leaf=min_samples_leaf; self.max_thresholds=max_thresholds
    def fit(self,X,y):
        self.X=np.asarray(X,float); self.y=np.asarray(y,float)
        self.root=self._grow(np.arange(len(y)),0); return self
    def _grow(self,idx,depth):
        yy=self.y[idx]; node={'value':float(yy.mean())}
        if depth>=self.max_depth or len(idx)<2*self.min_samples_leaf: return node
        parent=variance_impurity(yy); best=0.; bf=bt=bm=None
        for j in range(self.X.shape[1]):
            u=np.unique(self.X[idx,j])
            if len(u)<2: continue
            th=np.unique(np.quantile(u,np.linspace(0,1,self.max_thresholds+2)[1:-1])) if len(u)>self.max_thresholds else (u[:-1]+u[1:])/2
            for t in th:
                mask=self.X[idx,j]<=t; nl=int(mask.sum()); nr=len(idx)-nl
                if min(nl,nr)<self.min_samples_leaf: continue
                gain=parent-nl/len(idx)*variance_impurity(self.y[idx[mask]])-nr/len(idx)*variance_impurity(self.y[idx[~mask]])
                if gain>best: best,bf,bt,bm=gain,j,float(t),mask
        if bf is not None:
            node.update(feature=bf,threshold=bt,left=self._grow(idx[bm],depth+1),right=self._grow(idx[~bm],depth+1))
        return node
    def _one(self,row,node):
        while 'feature' in node: node=node['left'] if row[node['feature']]<=node['threshold'] else node['right']
        return node['value']
    def predict(self,X): return np.array([self._one(r,self.root) for r in np.asarray(X,float)])

class ScratchKNNRegressor:
    def __init__(self,k=5,weights='distance'): self.k=k; self.weights=weights
    def fit(self,X,y): self.X=np.asarray(X,float); self.y=np.asarray(y,float); return self
    def predict(self,X):
        out=[]
        for row in np.asarray(X,float):
            d=np.sqrt(np.sum((self.X-row)**2,axis=1)); ix=np.argsort(d)[:self.k]
            w=1/np.maximum(d[ix],1e-12) if self.weights=='distance' else np.ones(len(ix))
            out.append(np.sum(w*self.y[ix])/w.sum())
        return np.asarray(out)

class ScratchLinearSVR:
    def __init__(self,C=.02,epsilon=2.,lr=.01,epochs=5000):
        self.C=C; self.epsilon=epsilon; self.lr=lr; self.epochs=epochs
    def fit(self,X,y):
        X=np.asarray(X,float); y=np.asarray(y,float); n,p=X.shape
        self.w=np.zeros(p); self.b=float(y.mean())
        for epoch in range(self.epochs):
            err=X@self.w+self.b-y; active=np.abs(err)>self.epsilon
            gw=self.w+self.C*np.mean(np.sign(err[active,None])*X[active],axis=0) if active.any() else self.w
            gb=self.C*np.mean(np.sign(err[active])) if active.any() else 0.
            rate=self.lr/(1+epoch/1500)
            self.w-=rate*gw; self.b-=rate*gb
        return self
    def predict(self,X): return np.asarray(X,float)@self.w+self.b

class ScratchMLPRegressor:
    def __init__(self,hidden=12,lr=.01,epochs=3000,l2=.02,seed=42):
        self.hidden=hidden; self.lr=lr; self.epochs=epochs; self.l2=l2; self.seed=seed
    def fit(self,X,y):
        X=np.asarray(X,float); y=np.asarray(y,float).reshape(-1,1)
        rng=np.random.default_rng(self.seed); n,p=X.shape
        self.W1=rng.normal(0,np.sqrt(2/p),(p,self.hidden)); self.b1=np.zeros((1,self.hidden))
        self.W2=rng.normal(0,np.sqrt(1/self.hidden),(self.hidden,1)); self.b2=np.zeros((1,1))
        for epoch in range(self.epochs):
            a1=np.tanh(X@self.W1+self.b1); pred=a1@self.W2+self.b2; err=pred-y
            d2=2*err/n; gW2=a1.T@d2+self.l2*self.W2; gb2=d2.sum(axis=0,keepdims=True)
            d1=(d2@self.W2.T)*(1-a1**2); gW1=X.T@d1+self.l2*self.W1; gb1=d1.sum(axis=0,keepdims=True)
            rate=self.lr/(1+epoch/1000)
            self.W2-=rate*gW2; self.b2-=rate*gb2; self.W1-=rate*gW1; self.b1-=rate*gb1
        return self
    def predict(self,X):
        X=np.asarray(X,float); return (np.tanh(X@self.W1+self.b1)@self.W2+self.b2).ravel()

## 4. Library models and hyperparameter optimization

GridSearchCV uses five-fold KFold on the training partition and negative RMSE as the selection score. Preprocessing is inside each pipeline, so each fold learns its scaler and category encoder only from its training portion.

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
def make_preprocessor():
    return ColumnTransformer([
        ('num',StandardScaler(),num_cols),
        ('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat_cols)
    ])
library_pipelines={
 'SVR':Pipeline([('prep',make_preprocessor()),('model',SVR())]),
 'MLP':Pipeline([('prep',make_preprocessor()),('model',MLPRegressor(max_iter=2500,early_stopping=True,random_state=42))]),
 'Decision Tree':Pipeline([('prep',make_preprocessor()),('model',DecisionTreeRegressor(random_state=42))]),
 'KNN':Pipeline([('prep',make_preprocessor()),('model',KNeighborsRegressor())])
}
param_grids={
 'SVR':{'model__C':[1,10,50],'model__epsilon':[2,5,10],'model__kernel':['rbf','linear']},
 'MLP':{'model__hidden_layer_sizes':[(6,),(12,)],'model__alpha':[0.1,1.0],'model__learning_rate_init':[0.005,0.01]},
 'Decision Tree':{'model__max_depth':[1,2,3],'model__min_samples_leaf':[3,5,8]},
 'KNN':{'model__n_neighbors':[3,5,7,9],'model__weights':['uniform','distance']}
}
searches={}
for name,pipe in library_pipelines.items():
    gs=GridSearchCV(pipe,param_grids[name],cv=cv,scoring='neg_root_mean_squared_error',n_jobs=-1)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); gs.fit(X_train,y_train)
    searches[name]=gs
    print(name,gs.best_params_,'CV RMSE',-gs.best_score_)

## 5. Train from-scratch models and evaluate

The scratch models use the same training rows and encoded/scaled feature matrix as the library models. The tree is deliberately shallow for a small sample; KNN uses five neighbours; SVR and MLP are educational implementations of their core optimization ideas.

In [ ]:
scratch_models={
 'SVR':ScratchLinearSVR(C=.02,epsilon=2.,lr=.01,epochs=5000),
 'MLP':ScratchMLPRegressor(hidden=12,lr=.01,epochs=3000,l2=.02),
 'Decision Tree':ScratchTreeRegressor(max_depth=2,min_samples_leaf=5),
 'KNN':ScratchKNNRegressor(k=5,weights='distance')
}
def regression_metrics(y_true,y_pred):
    return {'MAE':mean_absolute_error(y_true,y_pred),
            'MSE':mean_squared_error(y_true,y_pred),
            'RMSE':np.sqrt(mean_squared_error(y_true,y_pred)),
            'R2':r2_score(y_true,y_pred)}
rows=[]; predictions={}
for name,gs in searches.items():
    pred=gs.best_estimator_.predict(X_test); predictions[('Library',name)]=pred
    rows.append({'Implementation':'Library','Model':name,**regression_metrics(y_test,pred)})
for name,model in scratch_models.items():
    model.fit(X_train_s,y_train); pred=model.predict(X_test_s)
    predictions[('From scratch',name)]=pred
    rows.append({'Implementation':'From scratch','Model':name,**regression_metrics(y_test,pred)})
results=pd.DataFrame(rows)
display(results.round(3))
baseline=np.repeat(y_train.mean(),len(y_test))
print('Test RMSE for training-mean baseline:',np.sqrt(mean_squared_error(y_test,baseline)))

## 6. Results visualization and residual analysis

In [ ]:
plot_labels=[f"{r.Model}\\n{r.Implementation}" for r in results.itertuples()]
plt.figure(figsize=(10,4)); plt.bar(plot_labels,results['RMSE'])
plt.ylabel('RMSE (percentage points)'); plt.title('Test RMSE comparison (lower is better)')
plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()

fig,axes=plt.subplots(2,4,figsize=(14,7),sharex=True,sharey=True)
for ax,(key,pred) in zip(axes.flat,predictions.items()):
    ax.scatter(y_test,pred,s=25); lo=min(y_test.min(),pred.min()); hi=max(y_test.max(),pred.max())
    ax.plot([lo,hi],[lo,hi],'--'); ax.set_title(f'{key[1]}\\n{key[0]}',fontsize=9)
    ax.set_xlabel('Actual growth'); ax.set_ylabel('Predicted growth')
plt.tight_layout(); plt.show()

# residuals for the best test RMSE model
best_row=results.loc[results.RMSE.idxmin()]
best_pred=predictions[(best_row.Implementation,best_row.Model)]
residuals=y_test.to_numpy()-best_pred
plt.figure(figsize=(7,4)); plt.scatter(best_pred,residuals); plt.axhline(0,linestyle='--')
plt.xlabel('Predicted growth (%)'); plt.ylabel('Residual (%)'); plt.title(f'Residual plot: {best_row.Model} ({best_row.Implementation})')
plt.tight_layout(); plt.show()

## 7. Interpretation, limitations and conclusion

The dataset is a cross-sectional snapshot of 100 companies, not a panel of repeated annual observations. The target contains an extreme high-growth observation; the small held-out set may therefore differ substantially from the training distribution. A negative test R² means the model is worse than predicting the training-target mean on this particular test partition. Compare this honestly with the CV RMSE; the discrepancy is evidence of instability and limited sample size, not a reason to hide the test result.

The from-scratch SVR is linear and therefore not equivalent to RBF/kernel SVR. The scratch MLP has one hidden layer and full-batch gradient descent. Scratch CART searches a capped set of candidate thresholds. These are transparent educational implementations, not optimized production solvers.

This work does not support causal claims or investment decisions. Future work should collect multiple years per company, use time-aware validation, include lagged financial statements available before the forecast date, compare with regularized linear regression and gradient boosting, and report uncertainty intervals. A time-based split is preferable once longitudinal data are available.

## References and research motivation

1. Pekin, S. & Şengül, A. (2024), *The good, the better and the challenging: Insights into predicting high-growth firms using machine learning*, Borsa Istanbul Review. https://doi.org/10.1016/j.bir.2024.12.001
2. Saito, M., Ohsato, T. & Yamanaka, S. (2021), *An empirical evaluation of machine learning performance in corporate sales growth prediction*, JSIAM Letters. https://doi.org/10.14495/jsiaml.13.25
3. Kapoor, S. & Narayanan, A. (2024), *Avoiding common machine learning pitfalls*, Patterns. https://doi.org/10.1016/j.patter.2024.101046
4. Scikit-learn, Cross-validation and avoiding data leakage: https://scikit-learn.org/stable/modules/cross_validation.html and https://scikit-learn.org/stable/common_pitfalls.html

**Dataset note:** `company_snapshot.csv` is embedded in this notebook so the notebook can be run in Colab without an additional upload.